# OLTP Transformation Testing

### Validate the incremental OLTP SQL before the transformation is moved into the production SQL runner and Airflow.

In [1]:
import sys

from pathlib import Path


project_root = Path.cwd().parent


# Add the project root to Python's module search path if necessary.
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


# Display the detected project root.
print("Project root:", project_root)

Project root: /Users/mac/Documents/netflix-data-engineering


## Import database connection

In [2]:
# Import the shared production PostgreSQL connection helper.
from src.database import get_etl_connection

##  Locate the staged titles batch

Find the latest real titles batch currently available in staging.

In [3]:
with get_etl_connection() as connection:

    with connection.cursor() as cursor:

        # Find the latest batch that loaded titles.csv.
        cursor.execute(
            """
            SELECT
                batch_id,
                file_name,
                rows_received,
                status
            FROM etl.batch_history
            WHERE file_name = 'titles.csv'
            ORDER BY batch_id DESC
            LIMIT 1;
            """
        )

        # Retrieve the latest matching batch.
        latest_titles_batch = cursor.fetchone()


# Stop if no titles batch exists.
assert latest_titles_batch is not None


# Extract the returned values.
titles_batch_id = latest_titles_batch[0]
titles_file_name = latest_titles_batch[1]
titles_rows_received = latest_titles_batch[2]
titles_batch_status = latest_titles_batch[3]


# Display the batch information.
print("Batch ID:", titles_batch_id)
print("File:", titles_file_name)
print("Rows received:", titles_rows_received)
print("Current status:", titles_batch_status)

Batch ID: 12
File: titles.csv
Rows received: 5850
Current status: RUNNING


## Capture the OLTP baseline

Record the current number of rows in the title table before running the incremental upsert.

In [4]:
with get_etl_connection() as connection:

    with connection.cursor() as cursor:

        # Count the current OLTP title records.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM title;
            """
        )

        # Store the baseline count.
        title_count_before = cursor.fetchone()[0]


# Display the baseline.
print("Title rows before upsert:", title_count_before)

Title rows before upsert: 5849


## Execute the incremental title upsert

Run the production OLTP title transformation for the staged titles batch using the real batch ID.

In [5]:
# Build the path to the production title upsert SQL file.
upsert_titles_sql_path = (
    project_root
    / "sql"
    / "02_oltp"
    / "01_upsert_titles.sql"
)


# Confirm that the production SQL file exists before trying to run it.
assert upsert_titles_sql_path.exists(), (
    f"SQL file not found: {upsert_titles_sql_path}"
)


# Read the SQL script into memory.
upsert_titles_sql = upsert_titles_sql_path.read_text(
    encoding="utf-8"
)


with get_etl_connection() as connection:

    # Open a cursor for executing the transformation.

    with connection.cursor() as cursor:

        # Execute the SQL and supply the real batch ID
        # to the %(batch_id)s placeholder inside the SQL file.
        cursor.execute(
            upsert_titles_sql,
            {
                "batch_id": titles_batch_id,
            },
        )

    # Commit the complete OLTP transformation.
    connection.commit()


print(
    f"PASS: batch {titles_batch_id} transformed into the OLTP title table."
)

PASS: batch 12 transformed into the OLTP title table.


## Test 4: Verify the OLTP result

Compare the title table before and after the incremental transformation.

In [6]:
# Open a fresh database connection for independent verification.
with get_etl_connection() as connection:

    # Open a cursor for the validation query.
    with connection.cursor() as cursor:

        # Count the OLTP title records after the upsert.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM title;
            """
        )

        # Store the resulting count.
        title_count_after = cursor.fetchone()[0]


print("Title rows before:", title_count_before)
print("Title rows after: ", title_count_after)

Title rows before: 5849
Title rows after:  5849


## Validate staged title coverage

Confirm that every valid title ID from the current staging batch exists in the OLTP title table.

In [7]:
with get_etl_connection() as connection:

    with connection.cursor() as cursor:

        # Count staged title IDs from this batch that are missing in OLTP.
        cursor.execute(
            """
            SELECT COUNT(*)

            FROM staging.titles_raw AS staging

            WHERE staging.batch_id = %s

              AND staging.id IS NOT NULL

              AND staging.id <> ''

              AND staging.title IS NOT NULL

              AND staging.title <> ''

              AND NOT EXISTS (
                  SELECT 1
                  FROM title AS oltp
                  WHERE oltp.title_id = staging.id
              );
            """,
            (titles_batch_id,),
        )

        # Retrieve the number of missing titles.
        missing_titles_count = cursor.fetchone()[0]


# Display the quality-check result.
print(
    "Valid staged titles missing from OLTP:",
    missing_titles_count,
)


# No valid staged title should be absent after the upsert.
assert missing_titles_count == 0


print(
    "PASS: all valid staged titles are present in OLTP."
)

Valid staged titles missing from OLTP: 0
PASS: all valid staged titles are present in OLTP.


## prove Idempotency 

Re-run the same batch and confirm that the title table does not gain duplicate rows.

In [8]:
with get_etl_connection() as connection:

    # Open a cursor for the rerun.
    with connection.cursor() as cursor:

        # Reprocess the same staged batch.
        cursor.execute(
            upsert_titles_sql,
            {
                "batch_id": titles_batch_id,
            },
        )

    # Commit the rerun.
    connection.commit()


# Count the OLTP titles after the second execution.

with get_etl_connection() as connection:

    with connection.cursor() as cursor:

        # Count the title table.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM title;
            """
        )

        # Store the count after rerunning the same batch.
        title_count_after_rerun = cursor.fetchone()[0]


# Display both post-transformation counts.
print("After first run:", title_count_after)
print("After rerun:    ", title_count_after_rerun)


# Reprocessing the same batch must not create more title records.
assert title_count_after_rerun == title_count_after


print(
    "PASS: title upsert is idempotent."
)

After first run: 5849
After rerun:     5849
PASS: title upsert is idempotent.


## Execute the incremental person upsert

Transform people from the staged credits batch into the normalised OLTP person table.

In [9]:
# Locate the production person-upsert SQL file.
upsert_people_sql_path = (
    project_root
    / "sql"
    / "02_oltp"
    / "02_upsert_people.sql"
)


# Confirm the SQL file exists.
assert upsert_people_sql_path.exists(), (
    f"SQL file not found: {upsert_people_sql_path}"
)


# Read the SQL file into memory.
upsert_people_sql = upsert_people_sql_path.read_text(
    encoding="utf-8"
)


# Find the most recent credits batch.
with get_etl_connection() as connection:

    with connection.cursor() as cursor:

        # Retrieve the latest credits.csv batch.
        cursor.execute(
            """
            SELECT batch_id
            FROM etl.batch_history
            WHERE file_name = 'credits.csv'
            ORDER BY batch_id DESC
            LIMIT 1;
            """
        )

        # Retrieve the matching batch.
        latest_credits_batch = cursor.fetchone()


# Confirm a credits batch exists.
assert latest_credits_batch is not None


# Extract the batch ID.
credits_batch_id = latest_credits_batch[0]


# Display the batch we're about to process.
print("Credits batch ID:", credits_batch_id)

Credits batch ID: 13


In [10]:
# Count people before the transformation.
with get_etl_connection() as connection:

    # Open a validation cursor.
    with connection.cursor() as cursor:

        # Count current OLTP people.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM person;
            """
        )

        person_count_before = cursor.fetchone()[0]


# Execute the person upsert.

with get_etl_connection() as connection:

    with connection.cursor() as cursor:

        # Process only the current credits batch.
        cursor.execute(
            upsert_people_sql,
            {
                "batch_id": credits_batch_id,
            },
        )

    connection.commit()


# Count people after the transformation.

with get_etl_connection() as connection:

    with connection.cursor() as cursor:

        # Count the OLTP person table again.
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM person;
            """
        )

        # Store the new count.
        person_count_after = cursor.fetchone()[0]


# Display the comparison.
print("People before:", person_count_before)
print("People after: ", person_count_after)


# Confirm the transformation completed.
print("PASS: staged people transformed into OLTP.")

People before: 54589
People after:  54589
PASS: staged people transformed into OLTP.


## Prove idempotency

In [11]:
# Run the same batch a second time.

with get_etl_connection() as connection:

    with connection.cursor() as cursor:

        # Execute the exact same credits batch again.
        cursor.execute(
            upsert_people_sql,
            {
                "batch_id": credits_batch_id,
            },
        )

    connection.commit()


# Count people after rerunning the same batch.

with get_etl_connection() as connection:

    with connection.cursor() as cursor:

        cursor.execute(
            """
            SELECT COUNT(*)
            FROM person;
            """
        )

        # Store the rerun count.
        person_count_after_rerun = cursor.fetchone()[0]


# Display the result.
print("After first run:", person_count_after)
print("After rerun:    ", person_count_after_rerun)


# Reprocessing the same batch must not create duplicate people.
assert person_count_after_rerun == person_count_after


# Confirm idempotency.
print("PASS: person upsert is idempotent.")

After first run: 54589
After rerun:     54589
PASS: person upsert is idempotent.


## Incremental genre upsert

Parse genres from the current titles batch and load unique values into the OLTP genre table.

In [12]:
upsert_genres_sql_path = (
    project_root
    / "sql"
    / "02_oltp"
    / "03_upsert_genres.sql"
)

# Confirm the SQL file exists.
assert upsert_genres_sql_path.exists()

# Read the production SQL.
upsert_genres_sql = upsert_genres_sql_path.read_text(
    encoding="utf-8"
)


# Capture the current genre count.

with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM genre;
            """
        )

        genre_count_before = cursor.fetchone()[0]


# Run the genre upsert for the current titles batch.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            upsert_genres_sql,
            {
                "batch_id": titles_batch_id,
            },
        )

    connection.commit()


# Capture the new genre count.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM genre;
            """
        )

        genre_count_after = cursor.fetchone()[0]


print("Genres before:", genre_count_before)
print("Genres after: ", genre_count_after)

print("PASS: genre upsert completed.")

Genres before: 19
Genres after:  19
PASS: genre upsert completed.


## Incremental country upsert

Parse production countries from the current titles batch and load unique values into the OLTP country table.

In [13]:
# Locate the production country upsert SQL.
upsert_countries_sql_path = (
    project_root
    / "sql"
    / "02_oltp"
    / "04_upsert_countries.sql"
)

assert upsert_countries_sql_path.exists()

# Read the SQL file.
upsert_countries_sql = upsert_countries_sql_path.read_text(
    encoding="utf-8"
)


# Capture the current country count.

with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM country;
            """
        )

        country_count_before = cursor.fetchone()[0]


# Run the incremental country upsert.

with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            upsert_countries_sql,
            {
                "batch_id": titles_batch_id,
            },
        )

    connection.commit()


# Capture the new count.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM country;
            """
        )

        country_count_after = cursor.fetchone()[0]


print("Countries before:", country_count_before)
print("Countries after: ", country_count_after)

print("PASS: country upsert completed.")

Countries before: 109
Countries after:  109
PASS: country upsert completed.


## Prove idempotency

In [14]:
# Re-run the same titles batch.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            upsert_countries_sql,
            {
                "batch_id": titles_batch_id,
            },
        )

    connection.commit()


# Count again after the rerun.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM country;
            """
        )

        country_count_after_rerun = cursor.fetchone()[0]


print("After first run:", country_count_after)
print("After rerun:    ", country_count_after_rerun)

assert country_count_after_rerun == country_count_after

print("PASS: country upsert is idempotent.")

After first run: 109
After rerun:     109
PASS: country upsert is idempotent.


In [22]:
# Run exactly the same bridge load again.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            sync_title_genres_sql,
            {
                "batch_id": titles_batch_id,
            },
        )

    connection.commit()


# Count the bridge rows again.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM title_genre;
            """
        )

        title_genre_count_after_rerun = cursor.fetchone()[0]


print("After first run:", title_genre_count_after)
print("After rerun:    ", title_genre_count_after_rerun)

assert title_genre_count_after_rerun == title_genre_count_after

print("PASS: title-genre bridge is idempotent.")

SyntaxError: cannot insert multiple commands into a prepared statement

## Title-country bridge synchronisation

Create the many-to-many relationships between titles and country for the current incremental batch and verify idempotency.

In [20]:
# Locate the production title-country bridge SQL.
sync_title_countries_sql_path = (
    project_root
    / "sql"
    / "02_oltp"
    / "06_sync_title_countries.sql"
)

# Confirm the SQL file exists.
assert sync_title_countries_sql_path.exists()

# Read the production SQL.
sync_title_countries_sql = sync_title_countries_sql_path.read_text(
    encoding="utf-8"
)


# Capture the current title-country bridge count.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM title_country;
            """
        )

        title_country_count_before = cursor.fetchone()[0]


# Synchronise title-country relationships for this batch.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            sync_title_countries_sql,
            {
                "batch_id": titles_batch_id,
            },
        )

    connection.commit()


# Capture the resulting bridge count.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM title_country;
            """
        )

        title_country_count_after = cursor.fetchone()[0]


print("Title-country rows before:", title_country_count_before)
print("Title-country rows after: ", title_country_count_after)

print("PASS: title-country bridge synchronised.")

Title-country rows before: 6528
Title-country rows after:  6528
PASS: title-country bridge synchronised.


## Prove Idempotency

In [ ]:
# Re-run the same title-country synchronisation.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            sync_title_countries_sql,
            {
                "batch_id": titles_batch_id,
            },
        )

    connection.commit()


# Count again after the rerun.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM title_country;
            """
        )

        title_country_count_after_rerun = cursor.fetchone()[0]


print("After first run:", title_country_count_after)
print("After rerun:    ", title_country_count_after_rerun)

assert title_country_count_after_rerun == title_country_count_after

print("PASS: title-country bridge is idempotent.")

After first run: 6528
After rerun:     6528
PASS: title-country bridge is idempotent.


## Incremental credit load

Load title/person credit relationships from the current credits batch and verify idempotency.

In [ ]:
# Locate the production credit SQL.
upsert_credits_sql_path = (
    project_root
    / "sql"
    / "02_oltp"
    / "07_upsert_credits.sql"
)

assert upsert_credits_sql_path.exists()

# Read the production SQL.
upsert_credits_sql = upsert_credits_sql_path.read_text(
    encoding="utf-8"
)


# Capture the current number of credits.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM public.credit;
            """
        )

        credit_count_before = cursor.fetchone()[0]


# Process the current credits batch.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            upsert_credits_sql,
            {
                "batch_id": credits_batch_id,
            },
        )

    connection.commit()


# Capture the resulting number of credits.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM public.credit;
            """
        )

        credit_count_after = cursor.fetchone()[0]


print("Credits before:", credit_count_before)
print("Credits after: ", credit_count_after)

print("PASS: credit transformation completed.")

Credits before: 77800
Credits after:  77800
PASS: credit transformation completed.


## Prove idempotency

In [ ]:
# Re-run the exact same credits batch.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            upsert_credits_sql,
            {
                "batch_id": credits_batch_id,
            },
        )

    connection.commit()


# Count again after the rerun.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM public.credit;
            """
        )

        credit_count_after_rerun = cursor.fetchone()[0]


print("After first run:", credit_count_after)
print("After rerun:    ", credit_count_after_rerun)

assert credit_count_after_rerun == credit_count_after

print("PASS: credit loading is idempotent.")

After first run: 77800
After rerun:     77800
PASS: credit loading is idempotent.


In [ ]:
# Check that every credit points to a valid title and person.
with get_etl_connection() as connection:
    with connection.cursor() as cursor:
        cursor.execute(
            """
            SELECT COUNT(*)
            FROM public.credit AS c

            LEFT JOIN public.title AS t
                ON t.title_id = c.title_id

            LEFT JOIN public.person AS p
                ON p.person_id = c.person_id

            WHERE t.title_id IS NULL
               OR p.person_id IS NULL;
            """
        )

        orphan_credit_count = cursor.fetchone()[0]


print("Orphan credits:", orphan_credit_count)

assert orphan_credit_count == 0

print("PASS: all credits reference valid titles and people.")

Orphan credits: 0
PASS: all credits reference valid titles and people.
